Загрузка конфиденциальных настроек

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


db_url = os.getenv("db_url") 
api_key = os.getenv("api_key")

Шаг 1. Импортирую необходимые библиотеки


In [2]:
import pandas as pd 
import requests 
import sqlalchemy 
import psycopg2


Шаг 2. Загружаю данные по велопарковкам

In [3]:

dataset_id = '916' 

all_data = [] # Список для накопления всех скачанных строк
top = 1000    # Максимально разрешенное API количество строк за раз
skip = 0      # Начальное смещение



while True:
    # Формируем ссылку с текущими параметрами пагинации
    url = f"https://apidata.mos.ru/v1/datasets/{dataset_id}/rows?api_key={api_key}&$top={top}&$skip={skip}"
    
    # Отправляем запрос
    response = requests.get(url)
    
    # Проверяем успешность запроса
    if response.status_code == 200:
        chunk = response.json()
        
        # Если сервер вернул пустой список, значит данные закончились
        if not chunk:
            break
            
        # Добавляем новую порцию данных в общий список
        all_data.extend(chunk)
        print(f"Загружено {len(all_data)} строк...")
        
        # Если пришло меньше 1000 строк, значит это последняя "страница"
        if len(chunk) < top:
            break
            
        # Увеличиваем смещение для следующего запроса на 1000
        skip += top
    else:
        print(f"Ошибка запроса: {response.status_code}")
        print(response.text)
        break

# создаем таблицу
if all_data:
    # Превращаем объединенный JSON в плоскую таблицу Pandas
    df = pd.json_normalize(all_data)
    
    print("\nДанные успешно загружены! Итоговый размер таблицы:", df.shape)
    
    
    display(df.head())
else:
    print("Данные не были загружены.")

Загружено 1000 строк...
Загружено 1697 строк...

Данные успешно загружены! Итоговый размер таблицы: (1697, 15)


,global_id,Number,Cells.ID,Cells.Name,Cells.AdmArea,Cells.District,Cells.Address,Cells.Photo,Cells.Capacity,Cells.DepartmentalAffiliation,Cells.ObjectOperOrgName,Cells.ObjectOperOrgPhone,Cells.Longitude_WGS84,Cells.Latitude_WGS84,Cells.global_id
0,2757725,1,32,Велосипедная парковка № 37124 «Гимназия № 1506»,Северо-Восточный административный округ,район Северное Медведково,"Широкая улица, дом 1А",4b2327b3-55af-4ff1-ab55-786c1d199557,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.650781,55.889628,2757725
1,2757729,2,36,Велосипедная парковка № 37106 «Центр образован...,Северо-Восточный административный округ,район Отрадное,"Отрадная улица, дом 11А",da0d57c7-03f4-47d7-b7e6-291d87f517bd,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.612922,55.857407,2757729
2,2757731,3,38,Велосипедная парковка № 37017 «Школа № 263»,Северо-Восточный административный округ,район Отрадное,"Отрадная улица, дом 11Б",21da1cde-ed77-4508-880a-bf29ff90db84,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.613732,55.856873,2757731
3,2757733,4,40,Велосипедная парковка № 37061 «Школа № 950»,Северо-Восточный административный округ,район Отрадное,"Отрадная улица, дом 1А",9e8b9d52-6939-478e-ae10-20dd9dffd969,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.604736,55.859911,2757733
4,2757735,5,2,Велосипедная парковка № 37026 «Школа № 301»,Северо-Восточный административный округ,Алтуфьевский район,"Алтуфьевское шоссе, дом 60Б",89259401-36c8-4b51-b6af-8752032c53bf,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.591231,55.883355,2757735


Шаг 3. Очищаю данные

In [4]:
# Очищаем названия колонок 
df.columns = df.columns.str.replace('Cells.', '')

# Удаляем лишние технические колонки
df = df.drop(columns=['global_id', 'Number'], errors='ignore')

# удаляем дубликаты строк, если они случайно затесались
df = df.drop_duplicates()

# Выводим количество пропусков по каждому столбцу
display(df.isna().sum())


display(df.head(3))

ID                         0
Name                       0
AdmArea                    0
District                   0
Address                    0
Photo                      0
Capacity                   0
DepartmentalAffiliation    0
ObjectOperOrgName          0
ObjectOperOrgPhone         0
Longitude_WGS84            0
Latitude_WGS84             0
dtype: int64

,ID,Name,AdmArea,District,Address,Photo,Capacity,DepartmentalAffiliation,ObjectOperOrgName,ObjectOperOrgPhone,Longitude_WGS84,Latitude_WGS84
0,32,Велосипедная парковка № 37124 «Гимназия № 1506»,Северо-Восточный административный округ,район Северное Медведково,"Широкая улица, дом 1А",4b2327b3-55af-4ff1-ab55-786c1d199557,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.650781,55.889628
1,36,Велосипедная парковка № 37106 «Центр образован...,Северо-Восточный административный округ,район Отрадное,"Отрадная улица, дом 11А",da0d57c7-03f4-47d7-b7e6-291d87f517bd,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.612922,55.857407
2,38,Велосипедная парковка № 37017 «Школа № 263»,Северо-Восточный административный округ,район Отрадное,"Отрадная улица, дом 11Б",21da1cde-ed77-4508-880a-bf29ff90db84,10,ДЕПАРТАМЕНТ ТРАНСПОРТА И РАЗВИТИЯ ДОРОЖНО-ТРАН...,ГОСУДАРСТВЕННОЕ КАЗЕННОЕ УЧРЕЖДЕНИЕ ГОРОДА МОС...,(495) 361-78-07,37.613732,55.856873


Шаг 4. Отправляю данные в PostgreSQL


In [5]:
from sqlalchemy import create_engine

# Задаем строку подключения к БД
engine = create_engine(db_url)

# Отправляем очищенный DataFrame  в PostgreSQL
df.to_sql(
    name='moscow_bike_parking', 
    con=engine,             
    if_exists='replace',    
)
index=False             